In [1]:
import pandas as pd
import requests

districts = pd.read_csv(
    "../datasets/district_coordinates.csv"
)

print(districts.head())

  District_Name  Latitude  Longitude
0       Lucknow   26.8393    80.9231
1        Kanpur   26.4652    80.3498
2          Agra   27.1833    78.0167
3      Varanasi   25.3167    83.0104
4     Prayagraj   25.4448    81.8432


In [2]:
lat = 26.8467
lon = 80.9462

url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M,RH2M,PRECTOTCORR&community=AG&longitude={lon}&latitude={lat}&start=20150101&end=20241231&format=JSON"

response = requests.get(url)

data = response.json()

print(data.keys())

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [3]:
weather_data = data['properties']['parameter']

print(weather_data.keys())

dict_keys(['T2M', 'RH2M', 'PRECTOTCORR'])


In [4]:
t2m = weather_data['T2M']
rh2m = weather_data['RH2M']
rain = weather_data['PRECTOTCORR']

df = pd.DataFrame({
    'Date': t2m.keys(),
    'T2M': t2m.values(),
    'RH2M': rh2m.values(),
    'PRECTOTCORR': rain.values()
})

print(df.head())

       Date    T2M   RH2M  PRECTOTCORR
0  20150101  15.55  50.47         0.28
1  20150102  17.12  73.35        19.02
2  20150103  18.07  82.78         5.55
3  20150104  17.08  67.61         0.29
4  20150105  14.10  47.95         0.03


In [5]:
df['Year'] = df['Date'].str[:4]

df['Year'] = df['Year'].astype(int)

In [6]:
yearly_weather = df.groupby('Year').agg({
    'T2M': 'mean',
    'RH2M': 'mean',
    'PRECTOTCORR': 'sum'
}).reset_index()

print(yearly_weather)

   Year        T2M       RH2M  PRECTOTCORR
0  2015  26.546466  46.651151       737.89
1  2016  26.391230  49.130410      1050.69
2  2017  26.231397  51.598548      1128.51
3  2018  25.947151  52.359288      1381.52
4  2019  25.403644  58.926575      1300.60
5  2020  24.768333  59.201803      1021.82
6  2021  25.170630  57.924219      1346.25
7  2022  25.452959  59.903397      1071.97
8  2023  25.545699  56.908521       953.08
9  2024  25.793033  56.153743      1163.97


In [7]:
yearly_weather['District_Name'] = 'Lucknow'

print(yearly_weather)

   Year        T2M       RH2M  PRECTOTCORR District_Name
0  2015  26.546466  46.651151       737.89       Lucknow
1  2016  26.391230  49.130410      1050.69       Lucknow
2  2017  26.231397  51.598548      1128.51       Lucknow
3  2018  25.947151  52.359288      1381.52       Lucknow
4  2019  25.403644  58.926575      1300.60       Lucknow
5  2020  24.768333  59.201803      1021.82       Lucknow
6  2021  25.170630  57.924219      1346.25       Lucknow
7  2022  25.452959  59.903397      1071.97       Lucknow
8  2023  25.545699  56.908521       953.08       Lucknow
9  2024  25.793033  56.153743      1163.97       Lucknow


In [8]:
all_weather = []

for index, row in districts.iterrows():

    district = row['District_Name']
    lat = row['Latitude']
    lon = row['Longitude']

    print(f"Processing {district}...")

    url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M,RH2M,PRECTOTCORR&community=AG&longitude={lon}&latitude={lat}&start=20150101&end=20241231&format=JSON"

    response = requests.get(url)

    data = response.json()

    weather_data = data['properties']['parameter']

    t2m = weather_data['T2M']
    rh2m = weather_data['RH2M']
    rain = weather_data['PRECTOTCORR']

    df = pd.DataFrame({
        'Date': t2m.keys(),
        'T2M': t2m.values(),
        'RH2M': rh2m.values(),
        'PRECTOTCORR': rain.values()
    })

    df['Year'] = df['Date'].str[:4]
    df['Year'] = df['Year'].astype(int)

    yearly = df.groupby('Year').agg({
        'T2M': 'mean',
        'RH2M': 'mean',
        'PRECTOTCORR': 'sum'
    }).reset_index()

    yearly['District_Name'] = district

    all_weather.append(yearly)

final_weather = pd.concat(all_weather)

print(final_weather.head())

Processing Lucknow...
Processing Kanpur...
Processing Agra...
Processing Varanasi...
Processing Prayagraj...
Processing Bareilly...
Processing Gorakhpur...
Processing Meerut...
Processing Aligarh...
Processing Jhansi...
   Year        T2M       RH2M  PRECTOTCORR District_Name
0  2015  26.667507  44.516877       629.19       Lucknow
1  2016  26.608907  46.551257       869.86       Lucknow
2  2017  26.421589  48.908712       960.07       Lucknow
3  2018  26.099068  50.693671      1216.01       Lucknow
4  2019  25.414192  57.753973      1122.86       Lucknow


In [9]:
final_weather.to_csv(
    "../datasets/weather/district_weather.csv",
    index=False
)

print("District weather dataset saved!")

District weather dataset saved!
